# Decision Tree Classifier

This notebook trains and evaluates a Decision Tree classifier for the CICIDS2017 network attack classification task.

A Decision Tree is a supervised model that predicts classes by splitting the data step by step based on feature values. It is easy to interpret because the decision path can be followed from the root node to a final leaf node.

The same datasets are used as in the other notebooks.

Load Data

In [27]:
import pandas as pd
import numpy as np

train = pd.read_csv("../train_data.csv")
validation = pd.read_csv("../val_data.csv")
test = pd.read_csv("../test_data.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print(train["Attack Type"].value_counts())

Train: (60000, 11)
Validation: (20000, 11)
Test: (20000, 11)
Attack Type
Normal Traffic    49849
DoS                4604
DDoS               3069
Port Scanning      2144
Brute Force         226
Web Attacks          55
Bots                 53
Name: count, dtype: int64


Features and label

In [28]:
LABEL_COL = "Attack Type"

FEATURE_COLS = [col for col in train.columns if col != LABEL_COL]

train = train.replace([np.inf, -np.inf], np.nan)
validation = validation.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

train_x = train[FEATURE_COLS]
train_y = train[LABEL_COL]

validation_x = validation[FEATURE_COLS]
validation_y = validation[LABEL_COL]

test_x = test[FEATURE_COLS]
test_y = test[LABEL_COL]

print("Features:")
print(FEATURE_COLS)
print("Number of features:", len(FEATURE_COLS))
print("Classes:", train_y.unique())

Features:
['Bwd Packet Length Std', 'Subflow Fwd Bytes', 'Flow Duration', 'Total Length of Fwd Packets', 'Init_Win_bytes_forward', 'Flow IAT Std', 'Active Mean', 'Bwd Packets/s', 'Fwd Packet Length Mean', 'Bwd Packet Length Min']
Number of features: 10
Classes: <StringArray>
[          'DDoS', 'Normal Traffic',  'Port Scanning',            'DoS',
    'Brute Force',    'Web Attacks',           'Bots']
Length: 7, dtype: str


Import libraries

- DecisionTreeClassifier is the model.
- SimpleImputer handles missing values.
- StandardScaler and MinMaxScaler are included for combinations that use SMOTE or dimensionality reduction.
- SMOTE is tested because minority attack classes are much smaller than Normal Traffic.
- PCA and LDA are tested as optional dimensionality reduction methods.
- GridSearchCV tests different parameter combinations.

In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit, ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

Train, validation and test datasets are already created, so PredefinedSplit is used.
- GridSearchCV trains on the training rows.
- GridSearchCV validates on the validation rows.
- The test set is only used once at the end.

In [30]:
# Combine train and validation sets for GridSearchCV
train_val_x = pd.concat([train_x, validation_x], axis=0)
train_val_y = pd.concat([train_y, validation_y], axis=0)

# Use the original validation set as the validation fold.
# -1 = training rows, 0 = validation rows
validation_fold = [-1] * len(train_x) + [0] * len(validation_x)
validation_split = PredefinedSplit(validation_fold)

Pipeline introduced
- Imputation
- Optional scaling
- Optional SMOTE
- Optional dimensionality reduction
- Decision Tree model

In [31]:
dt_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", "passthrough"),
    ("smote", "passthrough"),
    ("dimred", "passthrough"),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

Parameter grid defines different combinations
- Basic Decision Tree without scaling, SMOTE or dimensionality reduction
- Decision Tree with SMOTE
- Decision Tree with PCA or LDA
- Different split criteria, tree depths, leaf sizes and class-weight options

Scoring
- Best model is selected with accuracy to keep the comparison consistent with Naive Bayes.
- Macro precision, macro recall and macro F1 are also calculated because the classes are imbalanced.

In [32]:
dt_param_grid = [
    # Basic Decision Tree without SMOTE or dimensionality reduction
    {
        "scaler": ["passthrough"],
        "smote": ["passthrough"],
        "dimred": ["passthrough"],
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 5],
        "model__class_weight": [None, "balanced"]
    },

    # Decision Tree with SMOTE
    {
        "scaler": [StandardScaler(), MinMaxScaler()],
        "smote": [SMOTE(random_state=RANDOM_STATE, k_neighbors=3)],
        "dimred": ["passthrough"],
        "model__criterion": ["gini"],
        "model__max_depth": [10, 20],
        "model__min_samples_leaf": [1, 5],
        "model__class_weight": [None]
    },

    # Decision Tree with dimensionality reduction
    {
        "scaler": [StandardScaler()],
        "smote": ["passthrough"],
        "dimred": [
            PCA(n_components=5, random_state=RANDOM_STATE),
            LDA(n_components=2)
        ],
        "model__criterion": ["gini"],
        "model__max_depth": [10, 20],
        "model__min_samples_leaf": [1, 5],
        "model__class_weight": [None, "balanced"]
    }
]

# Define scoring metrics
scoring = {
    "accuracy": "accuracy",
    "macro_precision": make_scorer(precision_score, average="macro", zero_division=0),
    "macro_recall": make_scorer(recall_score, average="macro", zero_division=0),
    "macro_f1": make_scorer(f1_score, average="macro", zero_division=0)
}

number_of_combinations = sum(len(ParameterGrid(grid)) for grid in dt_param_grid)
print("Number of tested parameter combinations:", number_of_combinations)


Number of tested parameter combinations: 48


GridSearchCV trains and evaluates all parameter combinations
- The validation fold is fixed with PredefinedSplit.
- `refit="accuracy"` means the best parameter combination is refitted after the search.
- Training scores are returned so overfitting can be checked.

In [ ]:
# Perform GridSearchCV
grid_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    scoring=scoring,
    refit="accuracy",
    cv=validation_split,
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
    error_score="raise"
)

# Fit the model
grid_search.fit(train_val_x, train_val_y)


Print best validation parameters and validation accuracy

In [ ]:
print("Best Validation Parameters:", grid_search.best_params_)
print("Best Validation Accuracy:", grid_search.best_score_)

Best Validation Parameters: {'dimred': 'passthrough', 'model__class_weight': 'balanced', 'model__criterion': 'gini', 'model__max_depth': None, 'model__min_samples_leaf': 1, 'scaler': 'passthrough', 'smote': 'passthrough'}
Best Validation Accuracy: 0.99585


GridSearchCV results table for all tested combinations
- Validation accuracy
- Validation macro precision, recall and F1
- Training accuracy
- Tree and preprocessing parameters

In [ ]:
dt_results_table = pd.DataFrame(grid_search.cv_results_)

columns_to_display = [
    "rank_test_accuracy",
    "param_scaler",
    "param_smote",
    "param_dimred",
    "param_model__criterion",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__class_weight",
    "mean_test_accuracy",
    "mean_test_macro_precision",
    "mean_test_macro_recall",
    "mean_test_macro_f1",
    "mean_train_accuracy"
]

dt_results_table_display = dt_results_table[columns_to_display].sort_values(
    by="rank_test_accuracy"
).rename(columns={
    "rank_test_accuracy": "Rank",
    "param_scaler": "Scaler",
    "param_smote": "SMOTE",
    "param_dimred": "Dimensionality Reduction",
    "param_model__criterion": "Criterion",
    "param_model__max_depth": "Max Depth",
    "param_model__min_samples_leaf": "Min Samples Leaf",
    "param_model__class_weight": "Class Weight",
    "mean_test_accuracy": "Validation Accuracy",
    "mean_test_macro_precision": "Validation Macro Precision",
    "mean_test_macro_recall": "Validation Macro Recall",
    "mean_test_macro_f1": "Validation Macro F1",
    "mean_train_accuracy": "Train Accuracy"
})

dt_results_table_display

,Rank,Scaler,SMOTE,Dimensionality Reduction,Criterion,Max Depth,Min Samples Leaf,Class Weight,Validation Accuracy,Validation Macro Precision,Validation Macro Recall,Validation Macro F1,Train Accuracy
12,1,passthrough,passthrough,passthrough,gini,None,1,balanced,0.99585,0.887979,0.855082,0.870272,0.999583
4,2,passthrough,passthrough,passthrough,gini,20,1,NaN,0.99575,0.882009,0.852135,0.865421,0.999100
18,3,passthrough,passthrough,passthrough,entropy,None,1,balanced,0.99545,0.889500,0.893149,0.890377,0.999583
10,4,passthrough,passthrough,passthrough,entropy,20,1,NaN,0.99540,0.837805,0.810659,0.820999,0.998950
0,4,passthrough,passthrough,passthrough,gini,None,1,NaN,0.99540,0.886434,0.881946,0.883586,0.999583
6,6,passthrough,passthrough,passthrough,entropy,None,1,NaN,0.99520,0.868867,0.863027,0.865388,0.999583
7,7,passthrough,passthrough,passthrough,entropy,None,5,NaN,0.99485,0.867174,0.883995,0.874119,0.997333
2,8,passthrough,passthrough,passthrough,gini,10,1,NaN,0.99480,0.771923,0.746717,0.758057,0.996633
11,9,passthrough,passthrough,passthrough,entropy,20,5,NaN,0.99465,0.849533,0.846410,0.847294,0.997133
5,10,passthrough,passthrough,passthrough,gini,20,5,NaN,0.99460,0.847756,0.835420,0.840706,0.996750


Best model with test data
- Selected with validation data.
- Evaluated with separate test data.
- Macro scores are included because minority classes are important in attack detection.

In [ ]:
# Evaluate the best model on the test set
best_dt_model = grid_search.best_estimator_

# Make predictions
train_predictions = best_dt_model.predict(train_x)
validation_predictions = best_dt_model.predict(validation_x)
test_predictions = best_dt_model.predict(test_x)

# Calculate test metrics
test_accuracy = accuracy_score(test_y, test_predictions)
test_macro_precision = precision_score(test_y, test_predictions, average="macro", zero_division=0)
test_macro_recall = recall_score(test_y, test_predictions, average="macro", zero_division=0)
test_macro_f1 = f1_score(test_y, test_predictions, average="macro", zero_division=0)

# Calculate train and validation accuracy after the final refit
train_accuracy = accuracy_score(train_y, train_predictions)
validation_accuracy_after_refit = accuracy_score(validation_y, validation_predictions)

# Print results
print("Train Accuracy:", train_accuracy)
print("Validation Accuracy from GridSearchCV:", grid_search.best_score_)
print("Validation Accuracy after final refit:", validation_accuracy_after_refit)
print("Test Accuracy:", test_accuracy)
print("Test macro Precision:", test_macro_precision)
print("Test macro Recall:", test_macro_recall)
print("Test macro F1:", test_macro_f1)

# Print confusion matrix and classification report
class_labels = best_dt_model.named_steps["model"].classes_
print("Class order:", class_labels)
print("\nConfusion Matrix:\n", confusion_matrix(test_y, test_predictions, labels=class_labels))

print("\nClassification Report:\n", classification_report(test_y, test_predictions, zero_division=0))

Train Accuracy: 0.9995166666666667
Validation Accuracy from GridSearchCV: 0.99585
Validation Accuracy after final refit: 0.9997
Test Accuracy: 0.99565
Test macro Precision: 0.8637179974419442
Test macro Recall: 0.9065789150574116
Test macro F1: 0.8824989750341782
Class order: ['Bots' 'Brute Force' 'DDoS' 'DoS' 'Normal Traffic' 'Port Scanning'
 'Web Attacks']

Confusion Matrix:
 [[   12     0     0     0     6     0     0]
 [    0    73     0     1     1     0     0]
 [    0     0  1022     0     1     0     0]
 [    0     0     0  1519    16     0     0]
 [   12     1     3    23 16560     8     9]
 [    0     0     0     0     1   714     0]
 [    0     0     0     0     5     0    13]]

Classification Report:
                 precision    recall  f1-score   support

          Bots       0.50      0.67      0.57        18
   Brute Force       0.99      0.97      0.98        75
          DDoS       1.00      1.00      1.00      1023
           DoS       0.98      0.99      0.99      15

Confusion matrix as a table
- Rows are true classes.
- Columns are predicted classes.
- This makes it easier to see which attack types are confused with each other.

In [ ]:
dt_confusion_matrix = pd.DataFrame(
    confusion_matrix(test_y, test_predictions, labels=class_labels),
    index=class_labels,
    columns=class_labels
)

dt_confusion_matrix

,Bots,Brute Force,DDoS,DoS,Normal Traffic,Port Scanning,Web Attacks
Bots,12,0,0,0,6,0,0
Brute Force,0,73,0,1,1,0,0
DDoS,0,0,1022,0,1,0,0
DoS,0,0,0,1519,16,0,0
Normal Traffic,12,1,3,23,16560,8,9
Port Scanning,0,0,0,0,1,714,0
Web Attacks,0,0,0,0,5,0,13


Feature importance
- Decision Trees can show which input features were most useful.
- If PCA or LDA is selected, original feature importance is not directly interpretable.

In [ ]:
model_step = best_dt_model.named_steps["model"]
dimred_step = best_dt_model.named_steps["dimred"]

if dimred_step == "passthrough":
    dt_feature_importance = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "Importance": model_step.feature_importances_
    }).sort_values("Importance", ascending=False)

    dt_feature_importance
else:
    print("Feature importance is not directly interpretable because the best model used PCA or LDA.")

Decision Tree summary

Best Decision Tree model is selected by validation accuracy.

In [ ]:
dt_summary = pd.DataFrame([{
    "Model": "Decision Tree",
    "Best Parameters": grid_search.best_params_,
    "Train Accuracy": train_accuracy,
    "Validation Accuracy": grid_search.best_score_,
    "Validation Accuracy After Refit": validation_accuracy_after_refit,
    "Test Accuracy": test_accuracy,
    "Test Macro Precision": test_macro_precision,
    "Test Macro Recall": test_macro_recall,
    "Test Macro F1": test_macro_f1
}])

dt_summary

,Model,Best Parameters,Train Accuracy,Validation Accuracy,Validation Accuracy After Refit,Test Accuracy,Test Macro Precision,Test Macro Recall,Test Macro F1
0,Decision Tree,"{'dimred': 'passthrough', 'model__class_weight...",0.999517,0.99585,0.9997,0.99565,0.863718,0.906579,0.882499
